# Guia da Aula 04 — Métodos Avançados em Reinforcement Learning

**Disciplina:** Modelos de Aprendizagem por Reforço  
**Aula:** 04 — Métodos Avançados em RL  
**Formato:** Panorâmica — 9 notebooks, cada um cobre um subcampo diferente  
**Bibliotecas:** numpy, matplotlib, gymnasium, torch (torch opcional nos NB05 e NB06)

| | |
|---|---|
| **Aula** | Aula 04 — Métodos Avançados em Reinforcement Learning |
| **Notebook** | 00 — Guia da Aula |
| **Seções** | — |
| **Tempo de leitura** | ~5 min |
| **Tempo de execução** | ~1 min |

**Pré-requisitos:** Aulas 02 e 03 completas (Value-Based e Policy-Based Methods).

**Competências para o Desafio Final:** Orientação sobre os 9 notebooks da aula e validação do ambiente de execução.

### Recapitulando

A Aula 03 completou a jornada pelos métodos policy-based: de REINFORCE ao SAC, incluindo controle contínuo com DDPG. A limitação que essas abordagens ainda carregam: assumem interação *online* com o ambiente, agente único, e função de recompensa explícita. A Aula 04 abre as dimensões em que esse conjunto de suposições falha na prática — e apresenta as extensões que o campo desenvolveu para cada caso.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import rl_utils
rl_utils.info_versoes()
rl_utils.configurar_matplotlib()
rl_utils.definir_seeds(42)

Biblioteca           Versão
--------------------------------
gymnasium            1.0.0
torch                2.11.0+cu130
numpy                2.4.2
matplotlib           3.10.8
pandas               3.0.1
scikit-learn         1.8.0


## Bloco 1 — Contexto e pergunta central

As Aulas 02 e 03 cobriram o núcleo do RL moderno: métodos baseados em valor (Q-Learning, DQN)
e métodos baseados em política (REINFORCE, PPO, SAC). Esses algoritmos funcionam bem em ambientes
simples, com um único agente, interação online com o ambiente e recompensa bem definida.

Na prática, muitos problemas não se encaixam nesse molde:

- E se o agente **não puder interagir** com o ambiente durante o treino?
- E se houver **vários agentes** aprendendo ao mesmo tempo?
- E se a recompensa **não puder ser programada** diretamente — precisar ser inferida ou aprendida?
- E se o comportamento desejado for tão complexo que exige **decomposição hierárquica**?
- E se o objetivo for alinhar o comportamento de um **modelo de linguagem** com preferências humanas?

> **Pergunta central da Aula 04:**  
> "Como o RL se estende para além dos ambientes simples e dos agentes únicos?"

Esta aula é deliberadamente panorâmica. Cada notebook cobre um subcampo diferente com profundidade
conceitual e um exemplo executável — não uma implementação completa de todos os algoritmos.

## Bloco 2 — Mapa da aula e ordem de execução

Execute os notebooks **na ordem numérica** (00 → 08). Cada notebook é autocontido.

| # | Arquivo | Conteúdo | Ambiente |
|---|---|---|---|
| 00 | `aula04_00_guia.ipynb` | Este guia + validação de ambiente | — |
| 01 | `aula04_01_panorama_metodos_avancados.ipynb` | Visão geral dos subcampos; por que o RL clássico não basta | numpy, matplotlib |
| 02 | `aula04_02_model_based_rl.ipynb` | Modelo do ambiente, planejamento, Dyna-Q, world models | numpy, matplotlib |
| 03 | `aula04_03_multiagent_rl.ipynb` | Cooperação, competição, não-estacionariedade, CTDE | numpy, matplotlib |
| 04 | `aula04_04_offline_rl.ipynb` | *Batch RL*, extrapolation error, Conservative Q-Learning | numpy, matplotlib, gymnasium |
| 05 | `aula04_05_hierarchical_irl_behavior_cloning.ipynb` | Options, *behavior cloning*, IRL vs imitação | numpy, matplotlib, torch |
| 06 | `aula04_06_rlhf_reward_models.ipynb` | RLHF, reward models, alinhamento de LLMs | numpy, matplotlib, torch |
| 07 | `aula04_07_seguranca_etica_infra_avaliacao.ipynb` | *Reward hacking*, reprodutibilidade, infraestrutura experimental | numpy, matplotlib |
| 08 | `aula04_08_comparativo_final.ipynb` | Síntese dos 7 subcampos; quando usar cada abordagem; Desafio Final | numpy, matplotlib |

> **Atenção:** NB01 e NB06 são predominantemente conceituais — o código produz tabelas e visualizações
> ilustrativas, não treinamento completo de redes neurais profundas. Isso é intencional:
> a profundidade deste material está nas referências apontadas ao final de cada notebook.

## Como executar os notebooks

**Ambiente Python:** Python ≥ 3.9. Ative o ambiente virtual e instale as dependências:

```bash
source .venv/bin/activate
pip install -r requirements.txt
jupyter lab
```

**PettingZoo (opcional — NB03):** se disponível, o NB03 pode usar ambientes de jogos multiagente.
Caso não esteja instalado, o notebook usa uma simulação própria de dois agentes.

```bash
pip install pettingzoo
```

## Instalação das dependências

Se alguma biblioteca obrigatória estiver faltando, execute no terminal:

```bash
pip install numpy matplotlib gymnasium torch
```

Para o Notebook 03 (PettingZoo — opcional):

```bash
pip install pettingzoo
```

In [2]:
import sys
import importlib

print(f"Python: {sys.version}")
print()

bibliotecas = {
    "numpy":      "numpy",
    "matplotlib": "matplotlib",
    "gymnasium":  "gymnasium",
    "torch":      "torch",
}
opcionais = {"pettingzoo": "pettingzoo"}

print("Bibliotecas principais:")
for nome, modulo in bibliotecas.items():
    try:
        mod = importlib.import_module(modulo)
        versao = getattr(mod, "__version__", "ok")
        print(f"  {nome:<14} {versao}")
    except ImportError:
        print(f"  {nome:<14} NÃO INSTALADO")

print()
print("Bibliotecas opcionais:")
for nome, modulo in opcionais.items():
    try:
        mod = importlib.import_module(modulo)
        versao = getattr(mod, "__version__", "ok")
        print(f"  {nome:<14} {versao}")
    except ImportError:
        print(f"  {nome:<14} não instalado (NB03 usará simulação própria)")

print()
print("Ambiente validado. Pode prosseguir para o NB01.")

Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]

Bibliotecas principais:
  numpy          2.4.2
  matplotlib     3.10.8
  gymnasium      1.0.0
  torch          2.11.0+cu130

Bibliotecas opcionais:
  pettingzoo     1.26.1

Ambiente validado. Pode prosseguir para o NB01.


## Bloco 3 — Leitura dos resultados da validação

A célula acima imprime as versões instaladas. Interpretação:

- `numpy`, `matplotlib` — obrigatórios; sem eles nenhum notebook executa.
- `gymnasium` — necessário nos NB04 e NB07.
- `torch` — necessário nos NB05 e NB06 (redes neurais de behavior cloning e reward model).
- `pettingzoo` — opcional; o NB03 detecta automaticamente e usa fallback se ausente.

Se alguma biblioteca obrigatória estiver faltando:

```bash
pip install numpy matplotlib gymnasium torch
```

## Bloco 4 — A progressão pedagógica da aula

Cada notebook responde a uma limitação diferente do RL clássico. A mensagem central de cada um:

| Notebook | Limitação resolvida | Mensagem |
|---|---|---|
| NB01 | Visão geral | "O RL avançado não é um único algoritmo — é um mapa de problemas que os métodos clássicos não resolvem." |
| NB02 | Agente sem modelo do mundo | "Quando o agente aprende como o mundo funciona, ele pode planejar em vez de apenas reagir." |
| NB03 | Ambiente com múltiplos agentes | "Com múltiplos agentes, o ambiente nunca é estacionário — o comportamento dos outros é parte do problema." |
| NB04 | Impossibilidade de interação online | "Aprender sem interagir é possível, mas exige cuidado: o dataset fixo define os limites do que pode ser inferido." |
| NB05 | Comportamento complexo e recompensa implícita | "Comportamento complexo pode ser decomposto em subtarefas — e recompensas podem ser inferidas, não apenas programadas." |
| NB06 | Recompensa definida por preferências humanas | "O RLHF transforma preferências humanas em sinal de treino — e com isso traz os desafios do alinhamento." |
| NB07 | Recompensa mal especificada e reprodutibilidade | "Um agente que maximiza bem a métrica errada pode ser o pior resultado possível." |
| NB08 | Síntese e aplicabilidade | "Cada extensão do RL resolve um problema que o RL clássico não conseguia — conhecer o mapa é saber quando usar cada ferramenta." |

## Bloco 5 — Limites deste guia e próximo passo

Este notebook não ensina nenhum algoritmo. Sua função é orientar a navegação e confirmar
que o ambiente está configurado corretamente.

**O que este guia não cobre:**
- Implementações completas dos algoritmos (AlphaZero, MADDPG, CQL, GAIL) — para isso,
  consulte os papers nas referências de cada notebook.
- Treinamento em escala (GPU, clusters distribuídos) — a Aula 04 foca em conceitos,
  não em infraestrutura de produção.

**Próximo passo:** abra o `aula04_01_panorama_metodos_avancados.ipynb` e execute as células
na ordem. O NB01 apresenta o mapa de subcampos com três demonstrações executáveis que
conectam os problemas aos métodos que os resolvem.

## Conexão com sistemas de recomendação reais

Os métodos avançados desta aula respondem às limitações dos algoritmos das Aulas 02 e 03 quando aplicados a sistemas de recomendação em escala industrial:

| Subcampo | Limitação que resolve | Exemplo em recomendação |
|---|---|---|
| **Model-based RL** | Interação online cara ou lenta | Planejar recomendações futuras sem novas interações com o usuário real |
| **Multi-agent RL** | Vários sistemas competindo pelos mesmos itens | Múltiplos agentes de recomendação coexistindo no mesmo catálogo |
| **Offline RL** | Impossibilidade de experimentos online seguros | Aprender política de recomendação a partir de logs históricos sem expor usuários a políticas não testadas |
| **Hierarchical RL** | Horizonte de decisão muito longo | Planejar a sessão completa (nível alto) enquanto escolhe o próximo item (nível baixo) |
| **RLHF** | Recompensa difícil de programar explicitamente | Aprender o que "boa recomendação" significa a partir de feedback qualitativo de usuários |
| **Safety/ética** | Otimização de métrica errada | Detectar e corrigir *reward hacking* antes que o sistema amplifique vieses no catálogo |

O MovieLens é o fio condutor do curso — mas os problemas desta aula são o que separa um protótipo de laboratório de um sistema de recomendação em produção.

## Mapeamento para o Desafio Final

| Notebook | Subcampo | Competência central | Quando usar no Desafio Final |
|---|---|---|---|
| NB01 — Panorama | Visão geral | Identificar qual extensão do RL clássico resolve cada problema | Diagnóstico: classificar o problema antes de implementar |
| NB02 — Model-based | Planejamento | Planejar com modelo aprendido; Dyna-Q; world models | Quando o ambiente é caro de simular ou observar online |
| NB03 — MARL | Multiagente | Cooperação, competição, CTDE, não-estacionariedade | Qualquer cenário com múltiplos agentes ou stakeholders concorrentes |
| NB04 — Offline RL | Dados offline | Aprender de dataset fixo; extrapolation error; CQL | Quando logs históricos existem mas novos experimentos são proibidos |
| NB05 — Hierárquico + IRL | Hierarquia | Decomposição de tarefas; behavior cloning; recompensa inferida | Tarefas de longo horizonte ou recompensa implícita no comportamento humano |
| NB06 — RLHF | Alinhamento | Pipeline de reward model; KL penalty; alinhamento | Qualquer sistema onde a recompensa vem de preferências humanas |
| NB07 — Safety/ética | Segurança | Detectar reward hacking; reprodutibilidade; logging experimental | Antes de entregar qualquer sistema de RL para produção ou avaliação |
| NB08 — Síntese | Comparativo | Quando usar cada extensão do RL clássico | Seleção da abordagem final para o Desafio Final |

**Guia de decisão:** interação online possível? → Aulas 02–03. Apenas dados offline? → offline RL (NB04). Múltiplos agentes? → MARL (NB03). Recompensa implícita? → IRL/RLHF (NB05–06). Alto impacto? → Safety primeiro (NB07).

Os termos abaixo serão aprofundados em cada notebook; consulte aqui como referência rápida.

In [3]:
# Glossário dos termos técnicos deste notebook
rl_utils.exibir_glossario([
    'agent', 'environment', 'state', 'action', 'reward',
    'episode', 'policy', 'return', 'model-based RL', 'offline RL',
])

Termo (EN)       Tradução (PT)                Descrição
--------------------------------------------------------------------------------------------------------
action           ação                         Decisão tomada pelo agente em cada passo.
agent            agente                       Entidade que toma decisões no ambiente.
environment      ambiente                     Sistema com o qual o agente interage.
episode          episódio                     Sequência completa de interação do início ao estado terminal.
model-based RL   RL baseado em modelo         Aprendizado que usa um modelo do ambiente para planejamento.
offline RL       RL offline                   Aprendizado a partir de um dataset fixo sem interação nova com o ambiente.
policy           política                     π(a|s) — distribuição de probabilidade sobre ações dado o estado.
return           retorno                      Soma (descontada) de recompensas futuras a partir de um estado.
reward           recomp

## Leituras e referências

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2ª ed.). MIT Press. Disponível em: http://incompleteideas.net/book/the-book-2nd.html

- Levine, S., Kumar, A., Tucker, G., & Fu, J. (2020). Offline Reinforcement Learning: Tutorial, Review, and Perspectives on Open Problems. *arXiv:2005.01643*. Disponível em: https://arxiv.org/abs/2005.01643

- Gronauer, S., & Diepold, K. (2022). Multi-agent deep reinforcement learning: a survey. *Artificial Intelligence Review*, 55, 895–943. Disponível em: https://link.springer.com/article/10.1007/s10462-021-09996-w

- Ouyang, L., et al. (2022). Training language models to follow instructions with human feedback. *NeurIPS 2022*. Disponível em: https://arxiv.org/abs/2203.02155

- Barto, A. G., & Mahadevan, S. (2003). Recent advances in hierarchical reinforcement learning. *Discrete Event Dynamic Systems*, 13, 41–77.

- Amodei, D., et al. (2016). Concrete Problems in AI Safety. *arXiv:1606.06565*. Disponível em: https://arxiv.org/abs/1606.06565